
# Experimento de Machine Learning con datasets UCI

## Objetivo
Implementar un flujo experimental reproducible en Python utilizando algoritmos clásicos de Machine Learning:

- Random Forest
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)
- Naive Bayes
- Decision Tree

Datasets utilizados del repositorio UCI:

1. Dow Jones Index
2. Heart Disease
3. Default of Credit Card Clients
4. Banknote Authentication
5. Car Evaluation

## Nota importante
El dataset **Banknote Authentication** es un problema de clasificación binaria con target ya etiquetado.
El dataset **Dow Jones Index** se transforma a clasificación binaria usando la mediana del objetivo para mantener consistencia con las métricas de clasificación (Accuracy, Precision, Recall, F1, etc.).


In [1]:

# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

import os

print("Librerías importadas correctamente")


Librerías importadas correctamente


In [ ]:

# ============================================================
# DEFINICIÓN DE DATASETS
# ============================================================

datasets = {
    "Dow_Jones_Index": 312,
    "Heart_Disease": 45,
    "Default_Credit_Card": 350,
    "Banknote_Authentication": 267,
    "Car_Evaluation": 19
}

print(datasets)


{'Dow_Jones_Index': 312, 'Heart_Disease': 45, 'Default_Credit_Card': 350, 'Pedal_Me_Bicycle_Deliveries': 847, 'Car_Evaluation': 19}


In [ ]:

# ============================================================
# FLUJO EXPERIMENTAL COMPLETO
# ============================================================

results = []

models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "NaiveBayes": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42)
}

os.makedirs("confusion_matrices", exist_ok=True)

for dataset_name, dataset_id in datasets.items():

    print("\n" + "="*70)
    print(f"DATASET: {dataset_name}")
    print("="*70)

    # --------------------------------------------------------
    # Carga del dataset
    # --------------------------------------------------------
    try:
        dataset = fetch_ucirepo(id=dataset_id)
    except Exception as e:
        print(f"Advertencia: no se pudo cargar {dataset_name} (id={dataset_id}): {e}")
        continue

    X = dataset.data.features.copy()
    y = dataset.data.targets.copy()

    # Conversión target a vector
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:,0]

    # --------------------------------------------------------
    # Caso especial: Dow Jones -> convertir a clasificación binaria
    # --------------------------------------------------------
    if dataset_name == "Dow_Jones_Index":

        if y.dtype != object:
            threshold = y.median()
            y = (y > threshold).astype(int)

    # --------------------------------------------------------
    # Eliminar filas vacías
    # --------------------------------------------------------
    X = X.dropna(how="all")
    y = y.loc[X.index]

    # --------------------------------------------------------
    # Identificar columnas
    # --------------------------------------------------------
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    # --------------------------------------------------------
    # Preprocesamiento
    # --------------------------------------------------------
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    # --------------------------------------------------------
    # Codificación del target
    # --------------------------------------------------------
    le = LabelEncoder()
    y_encoded = le.fit_transform(y.astype(str))

    # --------------------------------------------------------
    # División entrenamiento / prueba
    # --------------------------------------------------------
    stratify_target = None
    if len(np.unique(y_encoded)) > 1:
        class_counts = np.bincount(y_encoded)
        if np.min(class_counts) >= 2:
            stratify_target = y_encoded
        else:
            print(f"Advertencia: no se puede estratificar {dataset_name} porque algunas clases tienen menos de 2 muestras. Se dividirá sin estratificación.")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.2,
        random_state=42,
        stratify=stratify_target
    )

    # --------------------------------------------------------
    # Entrenamiento y evaluación
    # --------------------------------------------------------
    for model_name, model in models.items():

        print(f"\nEntrenando: {model_name}")

        # Naive Bayes requiere matriz densa
        if model_name == "NaiveBayes":
            preprocessor_nb = ColumnTransformer(
                transformers=[
                    ("num", numeric_transformer, numeric_features),
                    ("cat", categorical_transformer, categorical_features)
                ],
                sparse_threshold=0
            )

            pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor_nb),
                ("classifier", model)
            ])

        else:
            pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", model)
            ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        cm = confusion_matrix(y_test, y_pred)

        # Guardar matriz
        cm_df = pd.DataFrame(cm)
        cm_path = f"confusion_matrices/{dataset_name}_{model_name}.csv"
        cm_df.to_csv(cm_path, index=False)

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": acc,
            "Balanced Accuracy": bal_acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1
        })


        print(f"Accuracy: {acc:.4f}")
        print(f"Balanced Accuracy: {bal_acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall: {rec:.4f}")
        print(f"F1-Score: {f1:.4f}")

print("\nProceso completado.")


DATASET: Dow_Jones_Index


ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 1

In [ ]:

# ============================================================
# EXPORTACIÓN DE RESULTADOS
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Dataset", "F1-Score"],
    ascending=[True, False]
)

results_df.to_csv("Resultados_Experimento.csv", index=False)

print(results_df.head())

print("\nArchivo exportado:")
print("Resultados_Experimento.csv")



# Conclusiones

## Flujo implementado

El experimento incluyó:

- Descarga automática desde UCI Repository
- Validación de datasets
- Limpieza de datos
- Imputación de valores faltantes
- Codificación de variables categóricas
- Escalamiento de variables numéricas
- División entrenamiento/prueba
- Entrenamiento de modelos
- Evaluación mediante métricas
- Exportación de resultados y matrices de confusión

## Métricas utilizadas

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix

## Reproducibilidad

Todos los experimentos utilizan `random_state=42` para garantizar reproducibilidad.
